# Sentiment Analysis using NLP Pipeline & ML Models


This project follows a complete NLP pipeline:

Raw Data → Text Preprocessing → Feature Engineering → Model Building → Model Evaluation → Comparison & Insights.

In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv("IMDB Dataset.csv")
df = df.sample(10000)
df.head()

,review,sentiment
35105,"I rented ""New Best Friend"" hoping for a movie ...",negative
25537,''The 40 Year Old Virgin'''made me laugh a lot...,positive
3477,This is bad movie. There is no denying it as m...,negative
14923,This movie really left me thinking ... but not...,negative
11481,"Worth the entertainment value of a rental, esp...",negative


In [3]:
df.shape

(10000, 2)

In [4]:
df.info()

<class 'pandas.DataFrame'>
Index: 10000 entries, 35105 to 42300
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype
---  ------     --------------  -----
 0   review     10000 non-null  str  
 1   sentiment  10000 non-null  str  
dtypes: str(2)
memory usage: 234.4 KB


In [5]:
df['sentiment'].value_counts()

sentiment
positive    5017
negative    4983
Name: count, dtype: int64

## NLP Preprocessing

In [6]:
!pip install nltk


[notice] A new release of pip is available: 26.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [7]:
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

In [8]:
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to C:\Users\K
[nltk_data]     Mallesh\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [9]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r'<.*?>', '', text)
    text = re.sub(r'[^a-zA-Z]', ' ', text)
    words = text.split()
    
    stop_words = set(stopwords.words('english'))
    words = [word for word in words if word not in stop_words]
    
    ps = PorterStemmer()
    words = [ps.stem(word) for word in words]
    
    return " ".join(words)

In [10]:
df['clean_review'] = df['review'].apply(clean_text)

In [11]:
df[['review', 'clean_review']].head()

,review,clean_review
35105,"I rented ""New Best Friend"" hoping for a movie ...",rent new best friend hope movi similar enjoy t...
25537,''The 40 Year Old Virgin'''made me laugh a lot...,year old virgin made laugh lot care consid sex...
3477,This is bad movie. There is no denying it as m...,bad movi deni much like tommi lee jone good po...
14923,This movie really left me thinking ... but not...,movi realli left think plot direct charact und...
11481,"Worth the entertainment value of a rental, esp...",worth entertain valu rental especi like action...


## Feature Engineering

In [12]:
!pip install scikit-learn


[notice] A new release of pip is available: 26.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [13]:
from sklearn.feature_extraction.text import CountVectorizer

In [14]:
cv = CountVectorizer(max_features=5000)
X_bow = cv.fit_transform(df['clean_review']).toarray()

In [15]:
X_bow.shape

(10000, 5000)

In [16]:
from sklearn.feature_extraction.text import TfidfVectorizer

In [17]:
tfidf = TfidfVectorizer(max_features=5000)
X_tfidf = tfidf.fit_transform(df['clean_review']).toarray()

In [18]:
X_tfidf.shape

(10000, 5000)

Feature extraction was performed using Bag of Words and TF-IDF techniques.

In [19]:
from sklearn.model_selection import train_test_split

X = X_tfidf   # You can also try X_bow later
y = df['sentiment']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

## Model Building

In [20]:
from sklearn.linear_model import LogisticRegression

lr = LogisticRegression()
lr.fit(X_train, y_train)

y_pred_lr = lr.predict(X_test)

In [21]:
from sklearn.naive_bayes import MultinomialNB

nb = MultinomialNB()
nb.fit(X_train, y_train)

y_pred_nb = nb.predict(X_test)

In [22]:
from sklearn.tree import DecisionTreeClassifier

dt = DecisionTreeClassifier(max_depth=10)
dt.fit(X_train, y_train)

y_pred_dt = dt.predict(X_test)

## Model Evaluation

In [23]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

In [24]:
print("Logistic Regression")
print("Accuracy:", accuracy_score(y_test, y_pred_lr))
print("Precision:", precision_score(y_test, y_pred_lr, pos_label='positive'))
print("Recall:", recall_score(y_test, y_pred_lr, pos_label='positive'))
print("F1 Score:", f1_score(y_test, y_pred_lr, pos_label='positive'))

Logistic Regression
Accuracy: 0.8765
Precision: 0.8670634920634921
Recall: 0.8855116514690983
F1 Score: 0.8761904761904762


In [25]:
print("\nNaive Bayes")
print("Accuracy:", accuracy_score(y_test, y_pred_nb))
print("Precision:", precision_score(y_test, y_pred_nb, pos_label='positive'))
print("Recall:", recall_score(y_test, y_pred_nb, pos_label='positive'))
print("F1 Score:", f1_score(y_test, y_pred_nb, pos_label='positive'))


Naive Bayes
Accuracy: 0.8495
Precision: 0.8565488565488566
Recall: 0.8348530901722391
F1 Score: 0.8455618265777322


In [26]:
print("\nDecision Tree")
print("Accuracy:", accuracy_score(y_test, y_pred_dt))
print("Precision:", precision_score(y_test, y_pred_dt, pos_label='positive'))
print("Recall:", recall_score(y_test, y_pred_dt, pos_label='positive'))
print("F1 Score:", f1_score(y_test, y_pred_dt, pos_label='positive'))


Decision Tree
Accuracy: 0.7035
Precision: 0.6483433734939759
Recall: 0.8723404255319149
F1 Score: 0.7438444924406048


## Comparison & Insights

Three models were trained: Logistic Regression, Naive Bayes, and Decision Tree. Their performance was evaluated using accuracy, precision, recall, and F1 score.

Logistic Regression performed best due to its ability to handle high-dimensional sparse data efficiently. Naive Bayes performed well but assumes feature independence. Decision Tree showed lower performance due to overfitting on text data.

## Summary of Findings

- The dataset was preprocessed using NLP techniques such as lowercasing, stopword removal, and stemming.
- Feature extraction was performed using Bag of Words and TF-IDF.
- Three models were trained: Logistic Regression, Naive Bayes, and Decision Tree.
- Logistic Regression achieved the best performance among all models.
- Naive Bayes performed well but assumes feature independence.
- Decision Tree showed lower performance due to overfitting.

Overall, TF-IDF with Logistic Regression provided the best results for sentiment classification.